<a href="https://colab.research.google.com/github/Pinkraaaa/Intro-to-AI-Group-9/blob/Khadijah-Preprocessing-and-data-collection/Group_9_CNN_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Cloning the project repository which contains the dataset as zipfiles and this colab notebook.

!git clone https://github.com/Pinkraaaa/Intro-to-AI-Group-9.git
%cd Intro-to-AI-Group-9

In [ ]:
#Extracting the three datasets splits from their respective zip files in ordr to use them

!unzip -q Potato_Test.zip
!unzip -q Potato_Train.zip
!unzip -q Potato_Validate.zip

In [ ]:
#Core imports: Pytorch, utilities and transforms.

import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [ ]:
#Make sure to match the directories after Pinkra makes the repo public
#Directory or paths matching each data split from the folders in the zip file above
train_dir = "Train"
val_dir   = "Valid"
test_dir  = "Test"

In [ ]:
# Resizing block

# Image training transformation pipeline:
# 1st Resizing
# 2nd Augmentation (made stringly due to the fact that we're training the model from scratch)
# 3rd Normalization
# We dont have any pretrained weights, so the model will need help generalising from a comparably small dataset (900 training images)

train_transform = transforms.Compose([
    transforms.Resize((128, 128)), # serves as image/input size, kept small to limit parameter counts
    transforms.RandomHorizontalFlip(), # changes the left/right orientation so the leaves have no fixed one.
    transforms.RandomVerticalFlip(), # same for up/down orientation
    transforms.RandomRotation(30), # serves to simulate random leaf/camera angles
    transforms.RandomResizedCrop(128, scale=(0.8, 1.0)), # serves to provide partial image views, so the model doesnt memorise the whole image
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # provides very mild lighting variation, with no hue shift (since the colour serves as the disease signal)
    transforms.ToTensor(), # converts the image to a PyTorch tensor
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # scaling the pixel values for stable training
])


# Image evaluation tranformation pipeline:
# 1st Resizing
# 2nd Normalization only
# No augmentation
# The images will be used for validation and test sets.
# We want to make sure we evaluate on realistic, unmodified images (simulating its functional use by users)

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)), # input suze
    transforms.ToTensor(), # conversion to PyTorch tensor
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) # scaling for stable training
])


In [ ]:
# Loading the dataset and dataloaders

# The ImageFolder method auto-labels the images by the respective folder names

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir, transform=eval_transform)
test_dataset  = datasets.ImageFolder(test_dir, transform=eval_transform)


# Wrapping the datasets in DataLoaders for batched, shuffles access
# Only the train_loader is shuffled, the validating and test order does not need to be shuffled

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [ ]:
#checking if it works
#This is just a sanity check, meant to confirm the output shape and class mapping of the pipeline

images, labels = next(iter(train_loader))
print(images.shape)          # expect: torch.Size([32, 3, 128, 128])
print(labels)
print(train_dataset.classes) # confirms which integer label maps to which class